In [ ]:
%%capture
import sys

if 'google.colab' in sys.modules:
    %pip install pyomo >/dev/null 2>/dev/null
    %pip install highspy >/dev/null 2>/dev/null

solver = 'appsi_highs' #or gurobi

import pyomo.environ as pyo
SOLVER = pyo.SolverFactory(solver)

assert SOLVER.available(), f"Solver {solver} is not available."

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving MASTER.csv to MASTER.csv


In [ ]:
# Load data directly from your CSV instead of .npy files

import pandas as pd

# 1) Read in the spreadsheet you uploaded (make sure the filename matches)
df = pd.read_csv('MASTER.csv')

# 2) Inspect column names to confirm
print("Available columns:", df.columns.tolist())

# 3) Replace the np.load(…) lines with these assignments,
#    matching the column names to your CSV headers:

activity_names = df['ActivityName'].tolist()     # e.g. column "ActivityName"
tt_means       = df['TravelTimeMean'].to_numpy()
tt_stds        = df['TravelTimeStd'].to_numpy()
tc_means       = df['TravelCostMean'].to_numpy()
tc_stds        = df['TravelCostStd'].to_numpy()
dur_means      = df['DurationMean'].to_numpy()
dur_stds       = df['DurationStd'].to_numpy()
costs_means    = df['CostMean'].to_numpy()
costs_stds     = df['CostStd'].to_numpy()


# Defining sets:
num_act = 73
park_ind = [57,58,59,60,61,66,67,68,69,70,71]
lunch_ind = [40,41,42,43,44,45,46,47,48]
dinner_ind = [49,50,51,52,53,54,55]
mor_hotel_ind = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]
eve_hotel_ind = [20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39]
days = [0,1,2]

# Defining parameters:
preferences = np.load('prefs.npy')
fatigues = np.load('fats.npy')
durs_all = np.maximum(0,np.random.normal(loc=dur_means, scale=dur_stds, size=(n,num_act)))
costs_all = np.maximum(0,np.random.normal(loc=costs_means, scale=costs_stds, size=(n,num_act)))
traveltimes_all = np.maximum(0,np.random.normal(loc=tt_means, scale=tt_stds, size=(n,num_act,num_act)))
travelcosts_all = np.maximum(0,np.random.normal(loc=tc_means, scale=tc_stds, size=(n,num_act,num_act)))

durs = []
costs = []
traveltimes = np.zeros((num_act,num_act))
travelcosts = np.zeros((num_act,num_act))

for i in range(num_act):
  durs.append(float(np.max(durs_all[:,i])))
  costs.append(float(np.max(costs_all[:,i])))

for i in range(num_act):
  for j in range(num_act):
    traveltimes[i,j] = float(np.max(traveltimes_all[:,i,j]))
    travelcosts[i,j] = float(np.max(travelcosts_all[:,i,j]))

alpha = 1
gamma = 0
budget = 600*2  #$600 per person, all prices have been calculated for two travelers

breakpoints = [0,1,2,3,4,5,6]
big_m = 1000

Available columns: ['Index', 'Name', 'Location', 'Duration (hrs) - avg', 'Duration (hrs) - std', 'Preference ', 'Cost - avg', 'Cost - std', 'Fatigue Score', 'Unnamed: 9']


KeyError: 'ActivityName'

In [ ]:
import numpy as np

# Create concrete model
model = pyo.ConcreteModel("3 Day Road Trip")

# Define sets
model.I = pyo.Set(initialize=[i for i in range(num_act)])  # All activities
model.P = pyo.Set(initialize=park_ind)  # Park activities
model.L = pyo.Set(initialize=lunch_ind)  # Lunch options
model.R = pyo.Set(initialize=dinner_ind)  # Dinner options
model.M = pyo.Set(initialize=mor_hotel_ind)  # Morning hotels
model.E = pyo.Set(initialize=eve_hotel_ind)  # Evening hotels
model.D = pyo.Set(initialize=days)  # Days

model.K = pyo.Set(initialize=[0,1,2,3,4,5,6])

# Define correspondence between morning and evening hotels
hotel_pairs = {0: 20, 1: 21, 2: 22, 3: 23, 4: 24, 5: 25, 6: 26, 7: 27, 8: 28, 9: 29,
               10: 30, 11: 31, 12: 32, 13: 33, 14: 34, 15: 35, 16: 36, 17: 37, 18: 38, 19: 39}

#for j in range(n):
#Define parameters
model.p = pyo.Param(model.I, initialize=lambda model, i: preferences[i])
model.f = pyo.Param(model.I, initialize=lambda model, i: fatigues[i])
model.dur = pyo.Param(model.I, initialize=lambda model, i: durs[0][i])
model.c = pyo.Param(model.I, initialize=lambda model, i: costs[0][i])
model.tt = pyo.Param(model.I, model.I, initialize=lambda model, i,a: traveltimes[0][i][a])
model.tc = pyo.Param(model.I, model.I, initialize=lambda model, i,a: travelcosts[0][i][a])

model.b = pyo.Param(model.K, initialize=lambda model, k: breakpoints[k])

# Define decision variables
model.z = pyo.Var(model.I, model.D, domain=pyo.Binary)
model.s = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.e = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.t = pyo.Var(model.I, model.I, model.D, domain=pyo.Binary)

# Variables for linearized quadratic penalty
model.y = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Excessive fatigue
model.pen = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Penalty value
model.lam = pyo.Var(model.D, model.K, domain=pyo.NonNegativeReals)


# Objective function
def objective_rule(model):
  utility = sum(model.p[i] * model.z[i, d] for d in model.D for i in model.I if i not in model.M) + sum(model.p[i] * model.z[i, 1] for i in model.M)
  penalty = sum(model.pen[d] for d in model.D)
  return alpha * utility - gamma * penalty
model.objective = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

# Constraints
# 1. Meal and Hotel Requirements
def lunch_req_rule(model, d):
  return sum(model.z[i, d] for i in model.L) == 1
model.lunch_req = pyo.Constraint(model.D, rule=lunch_req_rule)

def dinner_req_rule(model, d):
  return sum(model.z[i, d] for i in model.R) == 1
model.dinner_req = pyo.Constraint(model.D, rule=dinner_req_rule)

def morning_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.M) == 1
model.morning_hotel_req = pyo.Constraint(model.D, rule=morning_hotel_req_rule)

def evening_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.E) == 1
model.evening_hotel_req = pyo.Constraint(model.D, rule=evening_hotel_req_rule)

# 2. Hotel Consistency
def hotel_consistency_rule(model, i, d):
  if i in model.M and d > 0:
    j = hotel_pairs[i]
    return model.z[i, d] == model.z[j, d-1]
  return pyo.Constraint.Skip
model.hotel_consistency = pyo.Constraint(model.I, model.D, rule=hotel_consistency_rule)

# 3. Meal Time Windows
def lunch_time_rule(model, i, d):
  return (12 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 14 * model.z[i, d])
model.lunch_time_lower = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[0])
model.lunch_time_upper = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[1])

def dinner_time_rule(model, i, d):
  return (18 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 20 * model.z[i, d])
model.dinner_time_lower = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[0])
model.dinner_time_upper = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[1])

# 4. Daily Time Limits
def start_time_rule(model, i, d):
  if i not in model.M:
    return model.s[i, d] >= 8 * model.z[i, d]
  return pyo.Constraint.Skip
model.start_time = pyo.Constraint(model.I, model.D, rule=start_time_rule)

def end_time_rule(model, i, d):
  return model.s[i, d] <= 22 * model.z[i, d]
model.end_time = pyo.Constraint(model.E, model.D, rule=end_time_rule)

# 5. Activity Duration
def end_upper_rule(model, i, d):
  return model.e[i, d] <= model.s[i, d] + model.dur[i]
model.end_upper = pyo.Constraint(model.I, model.D, rule=end_upper_rule)

def end_binary_rule(model, i, d):
  return model.e[i, d] <= big_m * model.z[i, d]
model.end_binary = pyo.Constraint(model.I, model.D, rule=end_binary_rule)

def end_lower_rule(model, i, d):
  return model.e[i, d] >= model.s[i, d] + model.dur[i] - big_m * (1 - model.z[i, d])
model.end_lower = pyo.Constraint(model.I, model.D, rule=end_lower_rule)

# 6. Park Visitation Requirement
def park_visit_rule(model):
  return sum(model.z[i, d] for d in model.D for i in model.P) >= 1
model.park_visit = pyo.Constraint(rule=park_visit_rule)

# 7. Activity Assignment Limit
def activity_limit_rule(model, i):
  return sum(model.z[i, d] for d in model.D) <= 1
model.activity_limit = pyo.Constraint(model.I, rule=activity_limit_rule)

# 8. Travel Timing
def travel_timing_rule(model, i, j, d):
  if i != j:
    return model.e[i, d] + model.tt[i, j] - big_m * (1 - model.t[i, j, d]) <= model.s[j, d]
  return pyo.Constraint.Skip
model.travel_timing = pyo.Constraint(model.I, model.I, model.D, rule=travel_timing_rule)

# 9. Travel Consistency
def out_travel_rule(model, i, d):
  if i not in model.E:  # Activities except evening hotels
    return sum(model.t[i, j, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.out_travel = pyo.Constraint(model.I, model.D, rule=out_travel_rule)

def in_travel_rule(model, i, d):
  if i not in model.M:  # Activities except morning hotels
    return sum(model.t[j, i, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.in_travel = pyo.Constraint(model.I, model.D, rule=in_travel_rule)

# 10. Budget Constraint
def budget_rule(model):
  activity_costs = sum(model.c[i] * model.z[i, d] for d in model.D for i in model.I)
  travel_costs = sum(model.tc[i, j] * model.t[i, j, d] for d in model.D for i in model.I for j in model.I if i != j)
  return activity_costs + travel_costs + 80 <= budget
model.budget = pyo.Constraint(rule=budget_rule)

# 11. Fatigue Constraints
def excess_fatigue_rule(model, d):
  return model.y[d] >= sum(model.f[i] * model.dur[i] * model.z[i, d] for i in model.I) - 10
model.excess_fatigue = pyo.Constraint(model.D, rule=excess_fatigue_rule)

# 12. Piecewise Approximation Constraints
def lambda_sum_rule(model, d):
  return sum(model.lam[d, k] for k in model.K) == 1
model.lambda_sum = pyo.Constraint(model.D, rule=lambda_sum_rule)

def y_value_rule(model, d):
  return model.y[d] == sum(model.lam[d, k] * model.b[k] for k in model.K)
model.y_value = pyo.Constraint(model.D, rule=y_value_rule)

def penalty_rule(model, d):
  return model.pen[d] == sum(model.lam[d, k] * (model.b[k]**2) for k in model.K)
model.penalty_value = pyo.Constraint(model.D, rule=penalty_rule)

def sos_rule(model, d):
  return list((k, model.lam[d, k]) for k in model.K)
model.sos = pyo.SOSConstraint(model.D, sos=2, rule=sos_rule)


solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model)

# Check solution status and extract results
if results.solver.status == pyo.SolverStatus.ok:
    print("Solution found!")
    print(f"Objective value: {pyo.value(model.objective)}")

    # Print schedule
    for d in model.D:
        print(f"Day {d+1} Schedule:")
        activities = [(i, pyo.value(model.s[i,d])) for i in model.I if pyo.value(model.z[i,d]) > 0.5]
        activities.sort(key=lambda x: x[1])  # Sort by start time

        for i, start_time in activities:
            end_time = pyo.value(model.e[i,d])
            act = activity_names[i]
            print(f"  Activity {act}: {start_time:.2f} - {end_time:.2f}")
else:
    print("No solution found.")

AttributeError: 'tuple' object has no attribute '_lb'

In [ ]:
import numpy as np

# Create concrete model
model = pyo.ConcreteModel("3 Day Road Trip")

# Define sets
model.I = pyo.Set(initialize=[i for i in range(num_act)])  # All activities
model.P = pyo.Set(initialize=park_ind)  # Park activities
model.L = pyo.Set(initialize=lunch_ind)  # Lunch options
model.R = pyo.Set(initialize=dinner_ind)  # Dinner options
model.M = pyo.Set(initialize=mor_hotel_ind)  # Morning hotels
model.E = pyo.Set(initialize=eve_hotel_ind)  # Evening hotels
model.D = pyo.Set(initialize=days)  # Days

model.K = pyo.Set(initialize=[0,1,2,3,4,5,6])

# Define correspondence between morning and evening hotels
hotel_pairs = {0: 20, 1: 21, 2: 22, 3: 23, 4: 24, 5: 25, 6: 26, 7: 27, 8: 28, 9: 29,
               10: 30, 11: 31, 12: 32, 13: 33, 14: 34, 15: 35, 16: 36, 17: 37, 18: 38, 19: 39}

#for j in range(n):
#Define parameters
model.p = pyo.Param(model.I, initialize=lambda model, i: preferences[i])
model.f = pyo.Param(model.I, initialize=lambda model, i: fatigues[i])
model.dur = pyo.Param(model.I, initialize=lambda model, i: durs[0][i])
model.c = pyo.Param(model.I, initialize=lambda model, i: costs[0][i])
model.tt = pyo.Param(model.I, model.I, initialize=lambda model, i,a: traveltimes[0][i][a])
model.tc = pyo.Param(model.I, model.I, initialize=lambda model, i,a: travelcosts[0][i][a])

model.b = pyo.Param(model.K, initialize=lambda model, k: breakpoints[k])

# Define decision variables
model.z = pyo.Var(model.I, model.D, domain=pyo.Binary)
model.s = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.e = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.t = pyo.Var(model.I, model.I, model.D, domain=pyo.Binary)

# Variables for linearized quadratic penalty
#model.y = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Excessive fatigue
#model.pen = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Penalty value
#model.lam = pyo.Var(model.D, model.K, domain=pyo.NonNegativeReals)

# Add a binary variable to indicate which activity comes first
model.comes_first = pyo.Var(model.I, model.I, model.D, domain=pyo.Binary)

# Objective function
def objective_rule(model):
  utility = sum(model.p[i] * model.z[i, d] for d in model.D for i in model.I if i not in model.M) + sum(model.p[i] * model.z[i, 1] for i in model.M)
  #penalty = sum(model.pen[d] for d in model.D)
  return alpha * utility #- gamma * penalty
model.objective = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

# Constraints
# 1. Meal and Hotel Requirements
def lunch_req_rule(model, d):
  return sum(model.z[i, d] for i in model.L) == 1
model.lunch_req = pyo.Constraint(model.D, rule=lunch_req_rule)

def dinner_req_rule(model, d):
  return sum(model.z[i, d] for i in model.R) == 1
model.dinner_req = pyo.Constraint(model.D, rule=dinner_req_rule)

def morning_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.M) == 1
model.morning_hotel_req = pyo.Constraint(model.D, rule=morning_hotel_req_rule)

def evening_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.E) == 1
model.evening_hotel_req = pyo.Constraint(model.D, rule=evening_hotel_req_rule)

# 2. Hotel Consistency
def hotel_consistency_rule(model, i, d):
  if i in model.M and d > 0:
    j = hotel_pairs[i]
    return model.z[i, d] == model.z[j, d-1]
  return pyo.Constraint.Skip
model.hotel_consistency = pyo.Constraint(model.I, model.D, rule=hotel_consistency_rule)

# 3. Meal Time Windows
def lunch_time_rule(model, i, d):
  return (12 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 14 * model.z[i, d])
model.lunch_time_lower = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[0])
model.lunch_time_upper = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[1])

def dinner_time_rule(model, i, d):
  return (18 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 20 * model.z[i, d])
model.dinner_time_lower = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[0])
model.dinner_time_upper = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[1])

# 4. Daily Time Limits
def start_time_rule(model, i, d):
  if i not in model.M:
    return model.s[i, d] >= 8 * model.z[i, d]
  return pyo.Constraint.Skip
model.start_time = pyo.Constraint(model.I, model.D, rule=start_time_rule)

def end_time_rule(model, i, d):
  return model.s[i, d] <= 22 * model.z[i, d]
model.end_time = pyo.Constraint(model.I, model.D, rule=end_time_rule)

'''def morning_hotel_time_rule(model, i, d):
  return (8 * model.z[i, d] >= model.s[i, d])
model.morning_hotel_time = pyo.Constraint(model.M, model.D, rule=morning_hotel_time_rule)'''

# 5. Activity Duration
def end_upper_rule(model, i, d):
  return model.e[i, d] <= model.s[i, d] + model.dur[i]
model.end_upper = pyo.Constraint(model.I, model.D, rule=end_upper_rule)

def end_binary_rule(model, i, d):
  return model.e[i, d] <= big_m * model.z[i, d]
model.end_binary = pyo.Constraint(model.I, model.D, rule=end_binary_rule)

def end_lower_rule(model, i, d):
  return model.e[i, d] >= model.s[i, d] + model.dur[i] - big_m * (1 - model.z[i, d])
model.end_lower = pyo.Constraint(model.I, model.D, rule=end_lower_rule)

# 6. Park Visitation Requirement
def park_visit_rule(model):
  return sum(model.z[i, d] for d in model.D for i in model.P) >= 1
model.park_visit = pyo.Constraint(rule=park_visit_rule)

# 7. Activity Assignment Limit
def activity_limit_rule(model, i):
  return sum(model.z[i, d] for d in model.D) <= 1
model.activity_limit = pyo.Constraint(model.I, rule=activity_limit_rule)

# 8. Travel Timing
def travel_timing_rule(model, i, j, d):
  if i != j:
    return model.e[i, d] + model.tt[i, j] - big_m * (1 - model.t[i, j, d]) <= model.s[j, d]
  return pyo.Constraint.Skip
model.travel_timing = pyo.Constraint(model.I, model.I, model.D, rule=travel_timing_rule)

# 9. Travel Consistency
def out_travel_rule(model, i, d):
  if i not in model.E:  # Activities except evening hotels
    return sum(model.t[i, j, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.out_travel = pyo.Constraint(model.I, model.D, rule=out_travel_rule)

def in_travel_rule(model, i, d):
  if i not in model.M:  # Activities except morning hotels
    return sum(model.t[j, i, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.in_travel = pyo.Constraint(model.I, model.D, rule=in_travel_rule)

# 10. Budget Constraint
def budget_rule(model):
  activity_costs = sum(model.c[i] * model.z[i, d] for d in model.D for i in model.I)
  travel_costs = sum(model.tc[i, j] * model.t[i, j, d] for d in model.D for i in model.I for j in model.I if i != j)
  return activity_costs + travel_costs + 80 <= budget
model.budget = pyo.Constraint(rule=budget_rule)

# 11. Fatigue Constraints
'''def excess_fatigue_rule(model, d):
  return model.y[d] >= sum(model.f[i] * model.dur[i] * model.z[i, d] for i in model.I) - 10
model.excess_fatigue = pyo.Constraint(model.D, rule=excess_fatigue_rule)

# 12. Piecewise Approximation Constraints
def lambda_sum_rule(model, d):
  return sum(model.lam[d, k] for k in model.K) == 1
model.lambda_sum = pyo.Constraint(model.D, rule=lambda_sum_rule)

def y_value_rule(model, d):
  return model.y[d] == sum(model.lam[d, k] * model.b[k] for k in model.K)
model.y_value = pyo.Constraint(model.D, rule=y_value_rule)

def penalty_rule(model, d):
  return model.pen[d] == sum(model.lam[d, k] * (model.b[k]**2) for k in model.K)
model.penalty_value = pyo.Constraint(model.D, rule=penalty_rule)

def sos_rule(model, d):
  return list((k, model.lam[d, k]) for k in model.K)
model.sos = pyo.SOSConstraint(model.D, sos=2, rule=sos_rule)'''


solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model)

# Check solution status and extract results
if results.solver.status == pyo.SolverStatus.ok:
    print("Solution found!")
    print(f"Objective value: {pyo.value(model.objective)}")

    # Print schedule
    for d in model.D:
        print(f"Day {d+1} Schedule:")
        activities = [(i, pyo.value(model.s[i,d])) for i in model.I if pyo.value(model.z[i,d]) > 0.5]
        activities.sort(key=lambda x: x[1])  # Sort by start time

        for i, start_time in activities:
            end_time = pyo.value(model.e[i,d])
            act = activity_names[i]
            print(f" {i} {act}: {start_time:.2f} - {end_time:.2f}")
else:
    print("No solution found.")

In [ ]:
 if pyo.value(model.s[i,d]) < 24

import numpy as np

# Create concrete model
model = pyo.ConcreteModel("3 Day Road Trip")

# Define sets
model.I = pyo.Set(initialize=[i for i in range(num_act)])  # All activities
model.P = pyo.Set(initialize=park_ind)  # Park activities
model.L = pyo.Set(initialize=lunch_ind)  # Lunch options
model.R = pyo.Set(initialize=dinner_ind)  # Dinner options
model.M = pyo.Set(initialize=mor_hotel_ind)  # Morning hotels
model.E = pyo.Set(initialize=eve_hotel_ind)  # Evening hotels
model.D = pyo.Set(initialize=days)  # Days

model.K = pyo.Set(initialize=[0,1,2,3,4,5,6])

# Define correspondence between morning and evening hotels
hotel_pairs = {0: 20, 1: 21, 2: 22, 3: 23, 4: 24, 5: 25, 6: 26, 7: 27, 8: 28, 9: 29,
               10: 30, 11: 31, 12: 32, 13: 33, 14: 34, 15: 35, 16: 36, 17: 37, 18: 38, 19: 39}

#for j in range(n):
#Define parameters
model.p = pyo.Param(model.I, initialize=lambda model, i: preferences[i])
model.f = pyo.Param(model.I, initialize=lambda model, i: fatigues[i])
model.dur = pyo.Param(model.I, initialize=lambda model, i: durs[0][i])
model.c = pyo.Param(model.I, initialize=lambda model, i: costs[0][i])
model.tt = pyo.Param(model.I, model.I, initialize=lambda model, i,a: traveltimes[0][i][a])
model.tc = pyo.Param(model.I, model.I, initialize=lambda model, i,a: travelcosts[0][i][a])

model.b = pyo.Param(model.K, initialize=lambda model, k: breakpoints[k])

# Define decision variables
model.z = pyo.Var(model.I, model.D, domain=pyo.Binary)
model.s = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.e = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.t = pyo.Var(model.I, model.I, model.D, domain=pyo.Binary)

# Variables for linearized quadratic penalty
model.y = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Excessive fatigue
#model.pen = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Penalty value
#model.lam = pyo.Var(model.D, model.K, domain=pyo.NonNegativeReals)


# Objective function
def objective_rule(model):
  utility = sum(model.p[i] * model.z[i, d] for d in model.D for i in model.I if i not in model.M) + sum(model.p[i] * model.z[i, 1] for i in model.M)
  penalty = sum(model.y[d] for d in model.D)
  return alpha * utility - gamma * penalty
model.objective = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

# Constraints
# 1. Meal and Hotel Requirements
def lunch_req_rule(model, d):
  return sum(model.z[i, d] for i in model.L) == 1
model.lunch_req = pyo.Constraint(model.D, rule=lunch_req_rule)

def dinner_req_rule(model, d):
  return sum(model.z[i, d] for i in model.R) == 1
model.dinner_req = pyo.Constraint(model.D, rule=dinner_req_rule)

def morning_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.M) == 1
model.morning_hotel_req = pyo.Constraint(model.D, rule=morning_hotel_req_rule)

def evening_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.E) == 1
model.evening_hotel_req = pyo.Constraint(model.D, rule=evening_hotel_req_rule)

# 2. Hotel Consistency
def hotel_consistency_rule(model, i, d):
    if i in model.M and d > 0:
        j = hotel_pairs[i]
        return model.z[i, d] == model.z[j, d-1]
    return pyo.Constraint.Skip
model.hotel_consistency = pyo.Constraint(model.I, model.D, rule=hotel_consistency_rule)

# 3. Meal Time Windows
def lunch_time_rule(model, i, d):
  return (12 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 14 * model.z[i, d])
model.lunch_time_lower = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[0])
model.lunch_time_upper = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[1])

def dinner_time_rule(model, i, d):
  return (18 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 20 * model.z[i, d])
model.dinner_time_lower = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[0])
model.dinner_time_upper = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[1])

# 4. Daily Time Limits
def start_time_rule(model, i, d):
  if i not in model.M:
    return model.s[i, d] >= 8 * model.z[i, d]
  return pyo.Constraint.Skip
model.start_time = pyo.Constraint(model.I, model.D, rule=start_time_rule)

def end_time_rule(model, i, d):
  return model.s[i, d] <= 22 * model.z[i, d]
model.end_time = pyo.Constraint(model.E, model.D, rule=end_time_rule)

# 5. Activity Duration
def end_upper_rule(model, i, d):
  return model.e[i, d] <= model.s[i, d] + model.dur[i]
model.end_upper = pyo.Constraint(model.I, model.D, rule=end_upper_rule)

def end_binary_rule(model, i, d):
  return model.e[i, d] <= big_m * model.z[i, d]
model.end_binary = pyo.Constraint(model.I, model.D, rule=end_binary_rule)

def end_lower_rule(model, i, d):
  return model.e[i, d] >= model.s[i, d] + model.dur[i] - big_m * (1 - model.z[i, d])
model.end_lower = pyo.Constraint(model.I, model.D, rule=end_lower_rule)

# 6. Park Visitation Requirement
def park_visit_rule(model):
  return sum(model.z[i, d] for d in model.D for i in model.P) >= 1
model.park_visit = pyo.Constraint(rule=park_visit_rule)

# 7. Activity Assignment Limit
def activity_limit_rule(model, i):
  return sum(model.z[i, d] for d in model.D) <= 1
model.activity_limit = pyo.Constraint(model.I, rule=activity_limit_rule)

# 8. Travel Timing
def travel_timing_rule(model, i, j, d):
  if i != j:
    return model.e[i, d] + model.tt[i, j] - big_m * (1 - model.t[i, j, d]) <= model.s[j, d]
  return pyo.Constraint.Skip
model.travel_timing = pyo.Constraint(model.I, model.I, model.D, rule=travel_timing_rule)

# 9. Travel Consistency
def out_travel_rule(model, i, d):
  if i not in model.E:  # Activities except evening hotels
    return sum(model.t[i, j, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.out_travel = pyo.Constraint(model.I, model.D, rule=out_travel_rule)

def in_travel_rule(model, i, d):
  if i not in model.M:  # Activities except morning hotels
    return sum(model.t[j, i, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.in_travel = pyo.Constraint(model.I, model.D, rule=in_travel_rule)

def max_one_travel_rule1(model, j, d):
  return sum(model.t[j, i, d] for i in model.I if i != j) <= 1
model.max_one_travel1 = pyo.Constraint(model.I, model.D, rule=max_one_travel_rule1)

def max_one_travel_rule2(model, i, d):
  return sum(model.t[j, i, d] for j in model.I if i != j) <= 1
model.max_one_travel2 = pyo.Constraint(model.I, model.D, rule=max_one_travel_rule2)

def max_one_travel_rule3(model, j, d):
  return sum(model.t[i, j, d] for i in model.I if i != j) <= 1
model.max_one_travel3 = pyo.Constraint(model.I, model.D, rule=max_one_travel_rule3)

def max_one_travel_rule4(model, i, d):
  return sum(model.t[i, j, d] for j in model.I if i != j) <= 1
model.max_one_travel4 = pyo.Constraint(model.I, model.D, rule=max_one_travel_rule4)

def no_simultaneous_starts(model, i, j, d):
  if i < j:  # To avoid redundant constraints
    return model.s[i, d] + 0.01 <= model.s[j, d] + big_m * (2 - model.z[i, d] - model.z[j, d])
  return pyo.Constraint.Skip
model.no_simultaneous = pyo.Constraint(model.I, model.I, model.D, rule=no_simultaneous_starts)

# 10. Budget Constraint
def budget_rule(model):
  activity_costs = sum(model.c[i] * model.z[i, d] for d in model.D for i in model.I)
  travel_costs = sum(model.tc[i, j] * model.t[i, j, d] for d in model.D for i in model.I for j in model.I if i != j)
  return activity_costs + travel_costs + 80 <= budget
model.budget = pyo.Constraint(rule=budget_rule)

# 11. Fatigue Constraints
def excess_fatigue_rule(model, d):
  return model.y[d] >= sum(model.f[i] * model.dur[i] * model.z[i, d] for i in model.I) - 10
model.excess_fatigue = pyo.Constraint(model.D, rule=excess_fatigue_rule)

'''# 12. Piecewise Approximation Constraints
def lambda_sum_rule(model, d):
  return sum(model.lam[d, k] for k in model.K) == 1
model.lambda_sum = pyo.Constraint(model.D, rule=lambda_sum_rule)

def y_value_rule(model, d):
  return model.y[d] == sum(model.lam[d, k] * model.b[k] for k in model.K)
model.y_value = pyo.Constraint(model.D, rule=y_value_rule)

def penalty_rule(model, d):
  return model.pen[d] == sum(model.lam[d, k] * (model.b[k]**2) for k in model.K)
model.penalty_value = pyo.Constraint(model.D, rule=penalty_rule)

def sos_rule(model, d):
  return list((k, model.lam[d, k]) for k in model.K)
model.sos = pyo.SOSConstraint(model.D, sos=2, rule=sos_rule)'''


solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model)

# Check solution status and extract results
if results.solver.status == pyo.SolverStatus.ok:
    print("Solution found!")
    print(f"Objective value: {pyo.value(model.objective)}")

    # Print schedule
    for d in model.D:
        print(f"Day {d+1} Schedule:")
        activities = [(i, pyo.value(model.s[i,d])) for i in model.I if pyo.value(model.z[i,d]) > 0.5]
        activities.sort(key=lambda x: x[1])  # Sort by start time

        for i, start_time in activities:
            end_time = pyo.value(model.e[i,d])
            act = activity_names[i]
            print(f"  {act}: {start_time:.2f} - {end_time:.2f}")
else:
    print("No solution found.")

In [ ]:
hotel_pairs = {0: 20, 1: 21, 2: 22, 3: 23, 4: 24, 5: 25, 6: 26, 7: 27, 8: 28, 9: 29,
               10: 30, 11: 31, 12: 32, 13: 33, 14: 34, 15: 35, 16: 36, 17: 37, 18: 38, 19: 39}

hotel_pairs[2]

22

In [ ]:
20 + 10 <= 11

False

In [ ]:
import numpy as np

# Create concrete model
model = pyo.ConcreteModel("3 Day Road Trip")

# Define sets
model.I = pyo.Set(initialize=[i for i in range(num_act)])  # All activities
model.P = pyo.Set(initialize=park_ind)  # Park activities
model.L = pyo.Set(initialize=lunch_ind)  # Lunch options
model.R = pyo.Set(initialize=dinner_ind)  # Dinner options
model.M = pyo.Set(initialize=mor_hotel_ind)  # Morning hotels
model.E = pyo.Set(initialize=eve_hotel_ind)  # Evening hotels
model.D = pyo.Set(initialize=days)  # Days

model.K = pyo.Set(initialize=[0,1,2,3,4,5,6])

# Define correspondence between morning and evening hotels
hotel_pairs = {0: 20, 1: 21, 2: 22, 3: 23, 4: 24, 5: 25, 6: 26, 7: 27, 8: 28, 9: 29,
               10: 30, 11: 31, 12: 32, 13: 33, 14: 34, 15: 35, 16: 36, 17: 37, 18: 38, 19: 39}

#for j in range(n):
#Define parameters
model.p = pyo.Param(model.I, initialize=lambda model, i: preferences[i])
model.f = pyo.Param(model.I, initialize=lambda model, i: fatigues[i])
model.dur = pyo.Param(model.I, initialize=lambda model, i: durs[0][i])
model.c = pyo.Param(model.I, initialize=lambda model, i: costs[0][i])
model.tt = pyo.Param(model.I, model.I, initialize=lambda model, i,a: traveltimes[0][i][a])
model.tc = pyo.Param(model.I, model.I, initialize=lambda model, i,a: travelcosts[0][i][a])

model.b = pyo.Param(model.K, initialize=lambda model, k: breakpoints[k])

# Define decision variables
model.z = pyo.Var(model.I, model.D, domain=pyo.Binary)
model.s = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.e = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.t = pyo.Var(model.I, model.I, model.D, domain=pyo.Binary)

# Variables for linearized quadratic penalty
#model.y = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Excessive fatigue
#model.pen = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Penalty value
#model.lam = pyo.Var(model.D, model.K, domain=pyo.NonNegativeReals)


# Objective function
def objective_rule(model):
  utility = sum(model.p[i] * model.z[i, d] for d in model.D for i in model.I if i not in model.M) + sum(model.p[i] * model.z[i, 1] for i in model.M)
  #penalty = sum(model.pen[d] for d in model.D)
  return alpha * utility #- gamma * penalty
model.objective = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

# Constraints
# 1. Meal and Hotel Requirements
def lunch_req_rule(model, d):
  return sum(model.z[i, d] for i in model.L) == 1
model.lunch_req = pyo.Constraint(model.D, rule=lunch_req_rule)

def dinner_req_rule(model, d):
  return sum(model.z[i, d] for i in model.R) == 1
model.dinner_req = pyo.Constraint(model.D, rule=dinner_req_rule)

def morning_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.M) == 1
model.morning_hotel_req = pyo.Constraint(model.D, rule=morning_hotel_req_rule)

def evening_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.E) == 1
model.evening_hotel_req = pyo.Constraint(model.D, rule=evening_hotel_req_rule)

# 2. Hotel Consistency
def hotel_consistency_rule(model, i, d):
  if i in model.M and d > 0:
    j = hotel_pairs[i]
    return model.z[i, d] == model.z[j, d-1]
  return pyo.Constraint.Skip
model.hotel_consistency = pyo.Constraint(model.I, model.D, rule=hotel_consistency_rule)

# 3. Meal Time Windows
def lunch_time_rule(model, i, d):
  return (12 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 14 * model.z[i, d])
model.lunch_time_lower = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[0])
model.lunch_time_upper = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[1])

def dinner_time_rule(model, i, d):
  return (18 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 20 * model.z[i, d])
model.dinner_time_lower = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[0])
model.dinner_time_upper = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[1])

# 4. Daily Time Limits
def start_time_rule(model, i, d):
  if i not in model.M:
    return model.s[i, d] >= 8 * model.z[i, d]
  return pyo.Constraint.Skip
model.start_time = pyo.Constraint(model.I, model.D, rule=start_time_rule)

def end_time_rule(model, i, d):
  return model.s[i, d] <= 22 * model.z[i, d]
model.end_time = pyo.Constraint(model.I, model.D, rule=end_time_rule)

# 5. Activity Duration
def end_upper_rule(model, i, d):
  return model.e[i, d] <= model.s[i, d] + model.dur[i]
model.end_upper = pyo.Constraint(model.I, model.D, rule=end_upper_rule)

def end_binary_rule(model, i, d):
  return model.e[i, d] <= big_m * model.z[i, d]
model.end_binary = pyo.Constraint(model.I, model.D, rule=end_binary_rule)

def end_lower_rule(model, i, d):
  return model.e[i, d] >= model.s[i, d] + model.dur[i] - big_m * (1 - model.z[i, d])
model.end_lower = pyo.Constraint(model.I, model.D, rule=end_lower_rule)

# 6. Park Visitation Requirement
def park_visit_rule(model):
  return sum(model.z[i, d] for d in model.D for i in model.P) >= 1
model.park_visit = pyo.Constraint(rule=park_visit_rule)

# 7. Activity Assignment Limit
def activity_limit_rule(model, i):
  return sum(model.z[i, d] for d in model.D) <= 1
model.activity_limit = pyo.Constraint(model.I, rule=activity_limit_rule)

# 8. Travel Timing
def travel_timing_rule(model, i, j, d):
  if i != j:
    return model.e[i, d] + model.tt[i, j] - big_m * (1 - model.t[i, j, d]) <= model.s[j, d]
  return pyo.Constraint.Skip
model.travel_timing = pyo.Constraint(model.I, model.I, model.D, rule=travel_timing_rule)

# 9. Travel Consistency
def out_travel_rule(model, i, d):
  if i not in model.E:  # Activities except evening hotels
    return sum(model.t[i, j, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.out_travel = pyo.Constraint(model.I, model.D, rule=out_travel_rule)

def in_travel_rule(model, i, d):
  if i not in model.M:  # Activities except morning hotels
    return sum(model.t[j, i, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.in_travel = pyo.Constraint(model.I, model.D, rule=in_travel_rule)

# 10. Budget Constraint
def budget_rule(model):
  activity_costs = sum(model.c[i] * model.z[i, d] for d in model.D for i in model.I)
  travel_costs = sum(model.tc[i, j] * model.t[i, j, d] for d in model.D for i in model.I for j in model.I if i != j)
  return activity_costs + travel_costs + 80 <= budget
model.budget = pyo.Constraint(rule=budget_rule)

'''# 11. Fatigue Constraints
def excess_fatigue_rule(model, d):
  return model.y[d] >= sum(model.f[i] * model.dur[i] * model.z[i, d] for i in model.I) - 10
model.excess_fatigue = pyo.Constraint(model.D, rule=excess_fatigue_rule)

# 12. Piecewise Approximation Constraints
def lambda_sum_rule(model, d):
  return sum(model.lam[d, k] for k in model.K) == 1
model.lambda_sum = pyo.Constraint(model.D, rule=lambda_sum_rule)

def y_value_rule(model, d):
  return model.y[d] == sum(model.lam[d, k] * model.b[k] for k in model.K)
model.y_value = pyo.Constraint(model.D, rule=y_value_rule)

def penalty_rule(model, d):
  return model.pen[d] == sum(model.lam[d, k] * (model.b[k]**2) for k in model.K)
model.penalty_value = pyo.Constraint(model.D, rule=penalty_rule)

def sos_rule(model, d):
  return list((k, model.lam[d, k]) for k in model.K)
model.sos = pyo.SOSConstraint(model.D, sos=2, rule=sos_rule)'''


solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model)

# Check solution status and extract results
if results.solver.status == pyo.SolverStatus.ok:
    print("Solution found!")
    print(f"Objective value: {pyo.value(model.objective)}")

    # Print schedule
    for d in model.D:
        print(f"Day {d+1} Schedule:")
        activities = [(i, pyo.value(model.s[i,d])) for i in model.I if pyo.value(model.z[i,d]) > 0.5]
        activities.sort(key=lambda x: x[1])  # Sort by start time

        for i, start_time in activities:
            end_time = pyo.value(model.e[i,d])
            act = activity_names[i]
            print(f"  {i} {act}: {start_time:.2f} - {end_time:.2f}")
else:
    print("No solution found.")

KeyboardInterrupt: 

In [ ]:
import numpy as np

# Create concrete model
model = pyo.ConcreteModel("3 Day Road Trip")

# Define sets
model.I = pyo.Set(initialize=[i for i in range(num_act)])  # All activities
model.P = pyo.Set(initialize=park_ind)  # Park activities
model.L = pyo.Set(initialize=lunch_ind)  # Lunch options
model.R = pyo.Set(initialize=dinner_ind)  # Dinner options
model.M = pyo.Set(initialize=mor_hotel_ind)  # Morning hotels
model.E = pyo.Set(initialize=eve_hotel_ind)  # Evening hotels
model.D = pyo.Set(initialize=days)  # Days

model.K = pyo.Set(initialize=[0,1,2,3,4,5,6])

# Define correspondence between morning and evening hotels
hotel_pairs = {0: 20, 1: 21, 2: 22, 3: 23, 4: 24, 5: 25, 6: 26, 7: 27, 8: 28, 9: 29,
               10: 30, 11: 31, 12: 32, 13: 33, 14: 34, 15: 35, 16: 36, 17: 37, 18: 38, 19: 39}

#for j in range(n):
#Define parameters
model.p = pyo.Param(model.I, initialize=lambda model, i: preferences[i])
model.f = pyo.Param(model.I, initialize=lambda model, i: fatigues[i])
model.dur = pyo.Param(model.I, initialize=lambda model, i: durs[i])
model.c = pyo.Param(model.I, initialize=lambda model, i: costs[i])
model.tt = pyo.Param(model.I, model.I, initialize=lambda model, i,j: traveltimes[i][j])
model.tc = pyo.Param(model.I, model.I, initialize=lambda model, i,j: travelcosts[i][j])

# Define decision variables
model.z = pyo.Var(model.I, model.D, domain=pyo.Binary)
model.s = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.e = pyo.Var(model.I, model.D, domain=pyo.NonNegativeReals)
model.t = pyo.Var(model.I, model.I, model.D, domain=pyo.Binary)

# Variables for linearized quadratic penalty
model.y = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Excessive fatigue
model.pen = pyo.Var(model.D, domain=pyo.NonNegativeReals)  # Penalty value
model.lam = pyo.Var(model.D, model.K, domain=pyo.NonNegativeReals)


# Objective function
def objective_rule(model):
  utility = sum(model.p[i] * model.z[i, d] for d in model.D for i in model.I if i not in model.M) + sum(model.p[i] * model.z[i, 1] for i in model.M)
  #penalty = sum(model.pen[d] for d in model.D)
  return alpha * utility #- gamma * penalty
model.objective = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

# Constraints
def sun_lb_rule(model, d):
  return model.s[65,d] >= model.z[65,d]*6
model.sun_lb = pyo.Constraint(model.D, rule=sun_lb_rule)

def sun_ub_rule(model, d):
  return model.s[65,d] <= model.z[65,d]*6.5
model.sun_ub = pyo.Constraint(model.D, rule=sun_ub_rule)

# 1. Meal and Hotel Requirements
def lunch_req_rule(model, d):
  return sum(model.z[i, d] for i in model.L) == 1
model.lunch_req = pyo.Constraint(model.D, rule=lunch_req_rule)

def dinner_req_rule(model, d):
  return sum(model.z[i, d] for i in model.R) == 1
model.dinner_req = pyo.Constraint(model.D, rule=dinner_req_rule)

def morning_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.M) == 1
model.morning_hotel_req = pyo.Constraint(model.D, rule=morning_hotel_req_rule)

def evening_hotel_req_rule(model, d):
  return sum(model.z[i, d] for i in model.E) == 1
model.evening_hotel_req = pyo.Constraint(model.D, rule=evening_hotel_req_rule)

# 2. Hotel Consistency
def hotel_consistency_rule(model, i, d):
  j = hotel_pairs[i]
  return model.z[i, d] == model.z[j, d-1]
model.hotel_consistency = pyo.Constraint(model.M, [1,2], rule=hotel_consistency_rule)

# 3. Meal Time Windows
def lunch_time_rule(model, i, d):
  return (12 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 14 * model.z[i, d])
model.lunch_time_lower = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[0])
model.lunch_time_upper = pyo.Constraint(model.L, model.D, rule=lambda m, i, d: lunch_time_rule(m, i, d)[1])

def dinner_time_rule(model, i, d):
  return (18 * model.z[i, d] <= model.s[i, d], model.s[i, d] <= 20 * model.z[i, d])
model.dinner_time_lower = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[0])
model.dinner_time_upper = pyo.Constraint(model.R, model.D, rule=lambda m, i, d: dinner_time_rule(m, i, d)[1])

# 4. Daily Time Limits
def start_time_rule(model, i, d):
  if i not in model.M:
    return model.s[i, d] >= 8 * model.z[i, d]
  return pyo.Constraint.Skip
model.start_time = pyo.Constraint(model.I, model.D, rule=start_time_rule)

def end_time_rule(model, i, d):
  return model.s[i, d] <= 22 * model.z[i, d]
model.end_time = pyo.Constraint(model.E, model.D, rule=end_time_rule)

# 5. Activity Duration
def end_upper_rule(model, i, d):
  return model.e[i, d] <= model.s[i, d] + model.dur[i]
model.end_upper = pyo.Constraint(model.I, model.D, rule=end_upper_rule)

def end_binary_rule(model, i, d):
  return model.e[i, d] <= big_m * model.z[i, d]
model.end_binary = pyo.Constraint(model.I, model.D, rule=end_binary_rule)

def end_lower_rule(model, i, d):
  return model.e[i, d] >= model.s[i, d] + model.dur[i] - big_m * (1 - model.z[i, d])
model.end_lower = pyo.Constraint(model.I, model.D, rule=end_lower_rule)

# 6. Park Visitation Requirement
def park_visit_rule(model):
  return sum(model.z[i, d] for d in model.D for i in model.P) >= 1
model.park_visit = pyo.Constraint(rule=park_visit_rule)

# 7. Activity Assignment Limit
def activity_limit_rule(model, i):
  return sum(model.z[i, d] for d in model.D) <= 1
model.activity_limit = pyo.Constraint(model.I, rule=activity_limit_rule)

# 8. Travel Timing
def travel_timing_rule(model, i, j, d):
  if i != j:
    return model.e[i, d] + model.tt[i, j] - big_m * (1 - model.t[i, j, d]) <= model.s[j, d]
  return pyo.Constraint.Skip
model.travel_timing = pyo.Constraint(model.I, model.I, model.D, rule=travel_timing_rule)

# 9. Travel Consistency
def out_travel_rule(model, i, d):
  if i not in model.E:  # Activities except evening hotels
    return sum(model.t[i, j, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.out_travel = pyo.Constraint(model.I, model.D, rule=out_travel_rule)

def in_travel_rule(model, i, d):
  if i not in model.M:  # Activities except morning hotels
    return sum(model.t[j, i, d] for j in model.I if j != i) == model.z[i, d]
  return pyo.Constraint.Skip
model.in_travel = pyo.Constraint(model.I, model.D, rule=in_travel_rule)

# 10. Budget Constraint
def budget_rule(model):
  activity_costs = sum(model.c[i] * model.z[i, d] for d in model.D for i in model.I)
  travel_costs = sum(model.tc[i, j] * model.t[i, j, d] for d in model.D for i in model.I for j in model.I if i != j)
  return activity_costs + travel_costs + 80 <= budget
model.budget = pyo.Constraint(rule=budget_rule)

'''# 11. Fatigue Constraints
def excess_fatigue_rule(model, d):
  return model.y[d] >= sum(model.f[i] * model.dur[i] * model.z[i, d] for i in model.I) - 10
model.excess_fatigue = pyo.Constraint(model.D, rule=excess_fatigue_rule)

# 12. Piecewise Approximation Constraints
def lambda_sum_rule(model, d):
  return sum(model.lam[d, k] for k in model.K) == 1
model.lambda_sum = pyo.Constraint(model.D, rule=lambda_sum_rule)

def y_value_rule(model, d):
  return model.y[d] == sum(model.lam[d, k] * model.b[k] for k in model.K)
model.y_value = pyo.Constraint(model.D, rule=y_value_rule)

def penalty_rule(model, d):
  return model.pen[d] == sum(model.lam[d, k] * (model.b[k]**2) for k in model.K)
model.penalty_value = pyo.Constraint(model.D, rule=penalty_rule)

def sos_rule(model, d):
  return list((k, model.lam[d, k]) for k in model.K)
model.sos = pyo.SOSConstraint(model.D, sos=2, rule=sos_rule)'''


solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model)

# Check solution status and extract results
if results.solver.status == pyo.SolverStatus.ok:
    print("Solution found!")
    print(f"Objective value: {pyo.value(model.objective)}")

    # Print schedule
    for d in model.D:
        print(f"Day {d+1} Schedule:")
        activities = [(i, pyo.value(model.s[i,d])) for i in model.I if pyo.value(model.z[i,d]) > 0.5]
        activities.sort(key=lambda x: x[1])  # Sort by start time

        for i, start_time in activities:
            end_time = pyo.value(model.e[i,d])
            act = activity_names[i]
            print(f"  {i} {act}: {start_time:.2f} - {end_time:.2f}")
else:
    print("No solution found.")

Solution found!
Objective value: 208.0
Day 1 Schedule:
  36 Best Friends Roadhouse: 8.00 - 10.00
  68 South Kaibab 'Ooh Aah Point': 8.00 - 10.16
  46 Quesadilla Mobilla: 12.00 - 12.54
  54 Sunset Grill: 18.00 - 19.13
  18 Lake Powell Resort: 983.78 - 991.78
Day 2 Schedule:
  67 Devils Garden to Landscape Arch: 8.00 - 9.49
  69 Bright Angel Point: 8.00 - 9.02
  71 Coral Pink Sand Dunes: 8.00 - 8.93
  70 Best Friends Sanctuary Tour: 8.00 - 9.87
  64 Monument Valley Loop Drive: 8.00 - 10.17
  56 Angels Landing Hike: 8.00 - 12.47
  57 The Narrows (Bottom-Up): 8.00 - 14.71
  61 Scenic Drive to Rainbow Point: 8.00 - 9.41
  66 Delicate Arch Hike: 9.49 - 11.98
  43 Big John’s Texas BBQ: 12.00 - 12.76
  59 Emerald Pools Trail: 12.47 - 14.04
  16 Best Friends Roadhouse: 13.95 - 21.95
  52 Sego Restaurant: 18.00 - 19.83
  37 Parry Lodge: 22.00 - 24.00
  62 Horseshoe Bend Overlook: 987.49 - 989.01
  63 Lower Antelope Canyon Tour: 989.20 - 991.93
  60 Navajo Loop & Queens Garden: 989.98 - 991.93
  